# Looking at PUSH data and running it through a detector

In [1]:
import astropy.units as u
import numpy as np

from snewpy.models.ccsn import Wolfe_2023
from snewpy.flavor_transformation import AdiabaticMSW
from snewpy.neutrino import MixingParameters
from snewpy.models.base import PinchedModel, SupernovaModel

## Some plotting functions

In [2]:
import pylab as plt

In [3]:
def plot_quantity(x:u.Quantity, y:u.Quantity, xlabel=None, ylabel=None, **kwargs):
    """Plot the X vs Y array, with given axis labels, adding units"""

    #just in case we are passed bare np.arrays iwthout units
    x = u.Quantity(x)
    y = u.Quantity(y)
    
    if(len(x)==len(y)):
        plt.plot(x.value,y.value,**kwargs)
    else:
        plt.stairs(edges=x.value,values=y.value,**kwargs)
    if xlabel is not None:
        if(not x.unit.is_unity()):
            xlabel+=', '+x.unit._repr_latex_()
        plt.xlabel(xlabel)
    if ylabel is not None:
        if(not y.unit.is_unity()):
            ylabel+=', '+y.unit._repr_latex_()
        plt.ylabel(ylabel)
        
def plot_rate(rate, axis:str='time', **kwargs):
    if axis=='time':
        x = rate.time.to('s')
        y = rate.integrate_or_sum('energy').array.squeeze()
    elif axis=='energy':
        x = rate.energy.to('MeV')
        y = rate.integrate_or_sum('time').array.squeeze()
    else: 
        raise ValueError(f'axis="{axis}" should be one of "time","energy"')
    plot_quantity(x,y,xlabel=axis.capitalize(), ylabel='Event rate', **kwargs)


In [4]:
#a helper function to calculate total rate
from snewpy.flux import Container
def sum_rates(rates:list):
    res = sum([rate.array for rate in rates])
    rate = rates[0]#take first as an instance
    return Container(res,rate.flavor, rate.time, rate.energy, integrable_axes=rate._integrable_axes)

## Create the neutrino flux at Earth from the model

In [5]:
Wolfe_2023.param

{'progenitor_mass': <Quantity [10.8, 27.6, 28.2, 29. , 40. ] solMass>,
 'eos': ['SFHo', 'SFHx', 'DD2', 'BHB', 'TM1', 'NL3'],
 'callibration': ['calI']}

In [6]:
# prepare the neutrino flux from the model
model = Wolfe_2023(progenitor_mass=27.6*u.Msun,eos='SFHo') # SN model
transformation = AdiabaticMSW(MixingParameters('NORMAL')) # Desired flavor transformation

times    = model.get_time()
energies = np.linspace(0,60,601)<<u.MeV
distance = 10*u.kpc

#get the flux from the model
flux = model.get_flux(t=times, E=energies, distance=distance, flavor_xform=transformation)
fluence = flux.integrate('time')

s27.6_SFHo_calI_Wolfe_luminosity.h5


KeyError: "Unable to synchronously open object (object 'times' doesn't exist)"

In [ ]:
t = 50*u.ms

ispec = model.get_initial_spectra(t, energies)
ospec_nmo = model.get_transformed_spectra(t, energies, transformation)

fig, axes = plt.subplots(1,2, figsize=(12,5), sharex=True, sharey=True, tight_layout=True)

for i, spec in enumerate([ispec, ospec_nmo]):
    ax = axes[i]
    plt.sca(ax)
    spec.plot('energy')
    
    ax.set(title='Initial Spectra: $t = ${:.1f}'.format(t) if i==0 else 'Oscillated Spectra: $t = ${:.1f}'.format(t))
    ax.grid()
    ax.legend(loc='upper right', ncol=2, fontsize=16)

ax = axes[0]
ax.set(ylabel=r'flux, MeV')

fig.tight_layout();

## Using a detector config from SNOwGLoBES

In [ ]:
from snewpy.rate_calculator import RateCalculator

#load the RateCalculator object
rc = RateCalculator()

### List available detectors

In [ ]:
#list available detectors
list(rc.detectors)

### Read the detector you need

In [ ]:
#read the detector
det = rc.read_detector('scint20kt')
det

### Inspecting the detector

In [ ]:
#list all the channels
det.channels

### Running the rate calculation

In [ ]:
rates = det.run(fluence)
plot_rate(sum_rates(list(rates.values())), axis='energy', label='Total', lw=2, color='k')
for chan,rate in rates.items():
    plot_rate(rate, axis='energy', label=chan)
#plt.yscale('log')
#plt.ylim(1e-2)
plt.legend(ncols=3)
plt.ylabel(f'Events per {rate.energy.diff()[0]<<u.MeV}')
plt.xlim(0,50)
plt.show()

In [ ]:
events = rc.run(fluence, det, detector_effects=False)        
events_smeared = rc.run(fluence, det, detector_effects=True)
        
# Compute number of events in all interaction channels
total_events  = sum([chan.integrate_or_sum('energy').array.squeeze().value for chan in events.values()])        
total_events_smeared  = sum([chan.integrate_or_sum('energy').array.squeeze().value for chan in events_smeared.values()])

print("Total events in detector (with smearing):" , total_events_smeared)